# Dictionary Geometry — token-type concepts, co-activation, geometry baselines

Covers `PAPER_NOTES.md` Section 8 (footprint / token-type concepts) and Sections 9-11
(co-activation Gram spectrum, dictionary geometry, baselines). No new datasets — pure
analysis of the already-cached SAE dictionary + analysis-set latents, so this notebook
is intentionally short. Requires `rabbit_hull.ipynb` (Phase 1) to have been run first,
and reuses `classification_segmentation.ipynb`'s cached analysis activations/latents if
present (recomputes them here otherwise — same cache paths, so whichever notebook runs
first pays the one-time cost).

## 0. Setup

In [ ]:
import os, sys

# Anchors on markers unique to this repo so `import config` always finds THIS
# project's src/, regardless of how/where Colab's CWD ended up (upload, git
# clone, Drive mount, ...) — sys.path.append("src") alone only works if CWD
# already happens to be the repo root, which Colab doesn't guarantee.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "sae.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "rabbit_hull"),
    "/content/rabbit_hull",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the repo root (needs src/config.py and src/sae.py). "
        "cd into the rabbit_hull folder first (e.g. %cd /content/rabbit_hull), "
        "then re-run this cell."
    )

print("Repo root:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from config import CONFIG
import utils
import data_utils
import model_utils
import sae as sae_module
import geometry

print(CONFIG)

## 1. Load cached SAE checkpoint and analysis-set latents

In [ ]:
centroids_path = Path(CONFIG["cache_dir"]) / "centroids.pt"
sae_path = Path(CONFIG["cache_dir"]) / "checkpoints" / "sae.pt"
if not (centroids_path.exists() and sae_path.exists()):
    raise RuntimeError(
        "No cached centroids/SAE checkpoint found — run rabbit_hull.ipynb (Phase 1) first."
    )
centroids = torch.load(centroids_path, weights_only=True)
sae = sae_module.load_checkpoint(centroids, CONFIG)
dictionary = sae.dictionary().detach().cpu().numpy()  # D, (n_atoms, d_model)

analysis_latents_path = Path(CONFIG["cache_dir"]) / "latents" / "analysis_sparse.pt"
if analysis_latents_path.exists():
    print(f"Loading cached {analysis_latents_path.name}…")
    analysis_indices, analysis_values = torch.load(analysis_latents_path, weights_only=True)
else:
    val_dataset = data_utils.load_imagenet_val(CONFIG)
    analysis_images, _ = data_utils.load_analysis_images(val_dataset, CONFIG)
    model, processor = model_utils.load_model(CONFIG)
    analysis_acts = utils.cached(
        Path(CONFIG["cache_dir"]) / "activations" / "analysis.pt",
        lambda: model_utils.extract_activations(analysis_images, model, processor, CONFIG),
    )
    analysis_indices, analysis_values = utils.cached(
        analysis_latents_path, lambda: sae_module.encode_sparse(sae, analysis_acts, CONFIG)
    )

rng = np.random.RandomState(CONFIG["random_state"])

## 2. Token-type concepts (footprint)

`ω_i` = mean activation of concept `i` at each of the 261 token positions, averaged over
the analysis set (§8, exact formula). Concepts are labeled cls-only/reg-only/spatial-only
by footprint mass fraction.

In [ ]:
omega = geometry.footprint(analysis_indices, analysis_values, CONFIG["n_atoms"])
entropy = geometry.footprint_entropy(omega)
token_types = geometry.classify_token_types(omega)

for label in ("cls-only", "reg-only", "spatial-only", "mixed"):
    count = (token_types == label).sum()
    print(f"{label}: {count} concepts")

plt.hist(entropy, bins=50)
plt.xlabel("footprint entropy"); plt.ylabel("# concepts")
plt.title("Footprint entropy distribution (Fig 7/20)")
plt.show()

## 3. Co-activation spectrum

`G = Z^T Z`, compared against the random and shuffled baselines (Appendix E's exact
constructions). `rho` (random baseline's target density) is approximated as
`sparsity_k / n_atoms`, since we work from the sparse code representation rather than a
materialized dense `Z`.

In [ ]:
G = geometry.gram_matrix(analysis_indices, analysis_values, CONFIG["n_atoms"])
rho = CONFIG["sparsity_k"] / CONFIG["n_atoms"]
G_random = geometry.random_gram_baseline(G, rho, rng)
G_shuffled = geometry.shuffled_gram_baseline(G, rng)

plt.plot(geometry.spectrum(G), label="SAE")
plt.plot(geometry.spectrum(G_random), label="random")
plt.plot(geometry.spectrum(G_shuffled), label="shuffled")
plt.yscale("log")
plt.xlabel("component"); plt.ylabel("eigenvalue"); plt.legend()
plt.title("Co-activation (Z^T Z) spectrum vs. baselines (Section 9)")
plt.show()

## 4. Gram vs. geometric-similarity correlation

Paper reports r≈0.28 between off-diagonal `Z^T Z` and `D D^T` entries (Fig 11B).

In [ ]:
from scipy.stats import pearsonr

DDt = dictionary @ dictionary.T
iu = np.triu_indices(CONFIG["n_atoms"], k=1)
r, p = pearsonr(G[iu], DDt[iu])
print(f"corr(Z^T Z, D D^T): r={r:.4f}  p={p:.2e}  R^2={r**2:.4f}")

## 5. Dictionary geometry

`D D^T` inner-product histogram and singular-value spectrum, against the random-sphere
and coherence-minimized (approximate Grassmannian) baselines (Section 10, Appendix F).

In [ ]:
H_random = geometry.random_sphere_baseline(CONFIG["n_atoms"], CONFIG["d_model"], rng)
H_grassmannian = geometry.coherence_minimized_baseline(
    CONFIG["n_atoms"], CONFIG["d_model"], CONFIG["grassmannian_steps"], CONFIG["grassmannian_lr"], rng, CONFIG["device"]
)

d_norm = dictionary / np.linalg.norm(dictionary, axis=1, keepdims=True)
sae_inner = (d_norm @ d_norm.T)[iu]
random_inner = (H_random @ H_random.T)[iu]
grassmannian_inner = (H_grassmannian @ H_grassmannian.T)[iu]

plt.hist(sae_inner, bins=100, alpha=0.5, density=True, label="SAE")
plt.hist(random_inner, bins=100, alpha=0.5, density=True, label="random")
plt.hist(grassmannian_inner, bins=100, alpha=0.5, density=True, label="grassmannian-like")
plt.xlabel("pairwise inner product"); plt.legend()
plt.title("Dictionary pairwise inner products vs. baselines (Fig 10)")
plt.show()

plt.plot(np.linalg.svd(dictionary, compute_uv=False), label="SAE")
plt.plot(np.linalg.svd(H_random, compute_uv=False), label="random")
plt.plot(np.linalg.svd(H_grassmannian, compute_uv=False), label="grassmannian-like")
plt.yscale("log")
plt.xlabel("component"); plt.ylabel("singular value"); plt.legend()
plt.title("Dictionary singular-value spectrum vs. baselines (Section 10)")
plt.show()

## 6. Hoyer sparsity

In [ ]:
hoyer_sae = geometry.hoyer_score(dictionary)
hoyer_random = geometry.hoyer_score(H_random)
hoyer_grassmannian = geometry.hoyer_score(H_grassmannian)
gaussian_expectation = 1 - np.sqrt(2 / np.pi)

plt.hist(hoyer_sae, bins=50, alpha=0.5, density=True, label="SAE")
plt.hist(hoyer_random, bins=50, alpha=0.5, density=True, label="random")
plt.hist(hoyer_grassmannian, bins=50, alpha=0.5, density=True, label="grassmannian-like")
plt.axvline(gaussian_expectation, color="k", linestyle="--", label=f"Gaussian expectation ({gaussian_expectation:.3f})")
plt.xlabel("Hoyer sparsity"); plt.legend()
plt.title("Dictionary atom Hoyer sparsity (Section 10)")
plt.show()

## 7. Antipodal pairs

In [ ]:
pairs = geometry.antipodal_pairs(dictionary, threshold=CONFIG["antipodal_threshold"])
print(f"{len(pairs)} antipodal pairs (cos <= {CONFIG['antipodal_threshold']})")
for i, j, cos_theta in pairs[:10]:
    print(f"  atoms ({i}, {j}): cos={cos_theta:.4f}")

## What's next

This notebook covers `PAPER_NOTES.md` §8-11 (footprint/token-type concepts, co-activation
spectrum, dictionary geometry, baselines). The Grassmannian baseline is an approximate
gradient-descent frame-potential minimization, not the paper's exact TAAP algorithm — see
`README.md`'s differences table. Absolute concept counts (e.g. cls-only/reg-only) are
much smaller than the paper's since `n_atoms=1000` here vs. 32,000 there.

Continue with `mrh_analysis.ipynb` — position-embedding analysis (§12), MRH empirical
evidence (§14).